# 13 — IL-8 / CXCL8 across all conditions (focus decision point) — publication figure

**Fig 1:** CXCL8 (IL-8) log2FC across every available contrast, both studies.
**Fig 2:** HSPA5 standalone (correlated readout, ρ=0.92 with CXCL8 — not an IL-8 comparison).

Auto-discovers the A06 DE tables on disk (no filename guessing). Nature-style: Helvetica/Arial,
diverging colour by direction, significance by star + saturation, minimal chrome.
Reads existing DE tables only. Kernel: any pandas/matplotlib env.

In [1]:
import pandas as pd, numpy as np, re, os, glob
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

A04 = Path(os.environ.get('A04_PROJ', str(Path.home()/'Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate')))
A06 = Path(os.environ.get('A06_PROJ', str(Path.home()/'Projects_shared/epithelial_coinfection-model_WORKING')))
A04T, A06T = A04/'results/tables', A06/'analysis/DE'
FIG = A04/'results/figures'; FIG.mkdir(parents=True, exist_ok=True)

# ---- Nature-ish rcParams --------------------------------------------------
mpl.rcParams.update({
  'font.family':'sans-serif',
  'font.sans-serif':['Helvetica','Arial','DejaVu Sans'],
  'font.size':11,'axes.linewidth':0.8,'axes.edgecolor':'#444444',
  'axes.spines.top':False,'axes.spines.right':False,
  'xtick.color':'#444444','ytick.color':'#444444','text.color':'#222222',
  'axes.labelcolor':'#222222','figure.dpi':140,'savefig.dpi':300,'savefig.bbox':'tight',
})
print('A04 tables:',A04T.exists(),'| A06 DE:',A06T.exists())

A04 tables: True | A06 DE: True


## 0. Auto-discover which A06 contrasts actually exist on disk

In [2]:
# list the pydeseq2 pairwise tables present, prefer the *_annotated versions
a06_files = sorted(glob.glob(str(A06T/'pydeseq2_*_vs_*.csv')))
print('A06 pydeseq2 tables on disk:')
for f in a06_files: print('  ', Path(f).name)

def pretty_a06(fname):
    m=re.search(r'pydeseq2_([A-Za-z0-9]+)_vs_([A-Za-z0-9]+)', fname)
    if not m: return fname
    a,b=m.group(1),m.group(2)
    # Cd_vs_Co is the 'Cd-mono vs Cd-co' bracket from the sketch
    if {a,b}=={'Cd','Co'}: return 'Cd-mono vs Cd-co'
    return f'{a} vs {b}'

# build A06 spec from disk (dedup annotated/non-annotated to one per contrast)
seen=set(); A06_specs=[]
for f in a06_files:
    key=re.sub(r'_annotated','',Path(f).name)
    if key in seen: continue
    seen.add(key)
    # prefer annotated if it exists
    ann=f.replace('.csv','_annotated.csv') if '_annotated' not in f else f
    use=ann if Path(ann).exists() else f
    A06_specs.append((Path(use).name, pretty_a06(f)))
print('\nusing A06 contrasts:'); [print('  ',l,'<-',fn) for fn,l in A06_specs]

A06 pydeseq2 tables on disk:
   pydeseq2_Bt_vs_Cd.csv
   pydeseq2_Bt_vs_Cd_annotated.csv
   pydeseq2_Bt_vs_Co.csv
   pydeseq2_Bt_vs_Co_annotated.csv
   pydeseq2_Bt_vs_Mock.csv
   pydeseq2_Bt_vs_Mock_annotated.csv
   pydeseq2_Cd_vs_Co.csv
   pydeseq2_Cd_vs_Co_annotated.csv
   pydeseq2_Cd_vs_Mock.csv
   pydeseq2_Cd_vs_Mock_annotated.csv
   pydeseq2_Co_vs_Mock.csv
   pydeseq2_Co_vs_Mock_annotated.csv

using A06 contrasts:
   Bt vs Cd <- pydeseq2_Bt_vs_Cd_annotated.csv
   Bt vs Co <- pydeseq2_Bt_vs_Co_annotated.csv
   Bt vs Mock <- pydeseq2_Bt_vs_Mock_annotated.csv
   Cd-mono vs Cd-co <- pydeseq2_Cd_vs_Co_annotated.csv
   Cd vs Mock <- pydeseq2_Cd_vs_Mock_annotated.csv
   Co vs Mock <- pydeseq2_Co_vs_Mock_annotated.csv


[None, None, None, None, None, None]

In [ ]:
import pandas as pd, numpy as np, re, os, glob
from pathlib import Path

A04 = Path(os.environ.get('A04_PROJ', str(Path.home()/'Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate')))
A06 = Path(os.environ.get('A06_PROJ', str(Path.home()/'Projects_shared/epithelial_coinfection-model_WORKING')))
A04T, A06T = A04/'results/tables', A06/'analysis/DE'

# ---- discover A06 contrasts on disk (prefer *_annotated) ----
def pretty_a06(fname):
    m=re.search(r'pydeseq2_([A-Za-z0-9]+)_vs_([A-Za-z0-9]+)', fname)
    if not m: return fname
    a,b=m.group(1),m.group(2)
    if {a,b}=={'Cd','Co'}: return 'Cd-mono vs Cd-co'
    return f'{a} vs {b}'

a06_files = sorted(glob.glob(str(A06T/'pydeseq2_*_vs_*.csv')))
seen=set(); A06_specs=[]
for f in a06_files:
    key=re.sub(r'_annotated','',Path(f).name)
    if key in seen: continue
    seen.add(key)
    ann=f.replace('.csv','_annotated.csv') if '_annotated' not in f else f
    use=ann if Path(ann).exists() else f
    A06_specs.append((Path(use).name, pretty_a06(f)))

A04_specs=[
  ('DE_Cd_vs_Mock_0mM.csv','Cd vs Mock · 0mM'),
  ('DE_Cd_vs_Mock_5mM.csv','Cd vs Mock · 5mM'),
  ('DE_Cd_vs_Mock_10mM.csv','Cd vs Mock · 10mM'),
  ('DE_Butyrate_5v0_Cd.csv','But 5v0 · Cd'),
  ('DE_Butyrate_10v0_Cd.csv','But 10v0 · Cd'),
  ('DE_Butyrate_5v0_Mock.csv','But 5v0 · Mock'),
  ('DE_Butyrate_10v0_Mock.csv','But 10v0 · Mock'),
]

# ---- loaders ----
def sym_index(df):
    for col in ['gene_name','symbol','gene_symbol','GeneSymbol','Gene','gene']:
        if col in df.columns: df=df.set_index(col); break
    df.index=[re.sub(r'\..*','',str(i)) for i in df.index]
    return df
def load(dirpath,fname):
    f=Path(dirpath)/fname
    if not f.exists(): return None
    return sym_index(pd.read_csv(f,index_col=0))
def getval(d,g):
    if d is None or g not in d.index: return (np.nan,np.nan)
    r=d.loc[g]
    lfc=r.get('log2FoldChange', r.get('log2FC', np.nan))
    padj=r.get('padj', r.get('pvalue', np.nan))
    return float(lfc),float(padj)

def gather(gene):
    rows=[]
    for fn,label in A06_specs:
        d=load(A06T,fn)
        if d is None: d=load(A06T,fn.replace('_annotated',''))
        lfc,padj=getval(d,gene)
        rows.append(dict(label=label,study='co-infection',lfc=lfc,padj=padj))
    for fn,label in A04_specs:
        d=load(A04T,fn); lfc,padj=getval(d,gene)
        rows.append(dict(label=label,study='butyrate',lfc=lfc,padj=padj))
    return pd.DataFrame(rows).dropna(subset=['lfc']).drop_duplicates('label').reset_index(drop=True)

cx=gather('CXCL8'); print('CXCL8 contrasts:',len(cx)); print(cx.round(3).to_string())

## Publication-style plotting function

In [ ]:
def stars(p):
    if pd.isna(p): return 'n.s.'
    return '***' if p<1e-3 else '**' if p<1e-2 else '*' if p<0.05 else 'n.s.'

def pub_plot(df, gene, subtitle, fname, cmap_name='RdBu_r'):
    df=df.copy()
    df['s']=(df.study!='co-infection').astype(int)      # co-infection block first
    df=df.sort_values(['s']).reset_index(drop=True)
    n=len(df); x=np.arange(n)
    vmax=max(1.0, np.abs(df.lfc).max())
    norm=mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cmap=mpl.cm.get_cmap(cmap_name)
    # significance -> alpha (bold if significant, faded if n.s.)
    alphas=[1.0 if (not pd.isna(p) and p<0.05) else 0.42 for p in df.padj]
    colors=[cmap(norm(v)) for v in df.lfc]

    fig,ax=plt.subplots(figsize=(max(7.5,0.82*n+2.2),5.0))
    bars=ax.bar(x, df.lfc, color=colors, edgecolor='#333333', linewidth=0.7, width=0.72, zorder=3)
    for b,a in zip(bars,alphas): b.set_alpha(a)
    # star labels
    for xi,(_,r) in zip(x,df.iterrows()):
        va='bottom' if r.lfc>=0 else 'top'; off=0.10 if r.lfc>=0 else -0.10
        ax.text(xi, r.lfc+off, stars(r.padj), ha='center', va=va,
                fontsize=9, fontweight='bold', color='#222', zorder=4)
    ax.axhline(0,color='#444',lw=0.9,zorder=2)
    # block separator between the two studies
    nco=(df.s==0).sum()
    if 0<nco<n:
        ax.axvline(nco-0.5, color='#bbbbbb', ls=(0,(4,3)), lw=1.0, zorder=1)
        ylo,yhi=ax.get_ylim(); ax.set_ylim(ylo, yhi*1.18)   # headroom so labels don't collide
        yl=ax.get_ylim()[1]*0.93
        ax.text((nco-1)/2, yl, 'co-infection study', ha='center', va='top',
                fontsize=10, color='#3a5f8a', fontweight='bold')
        ax.text((nco+n-1)/2, yl, 'butyrate study', ha='center', va='top',
                fontsize=10, color='#a5551d', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(df.label, rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('log$_2$ fold-change', fontsize=11)
    ax.set_title(gene, fontsize=15, fontweight='bold', loc='left', pad=18)
    ax.text(0,1.02, subtitle, transform=ax.transAxes, fontsize=9.5, color='#666', va='bottom')
    ax.margins(x=0.02); ax.grid(axis='y', color='#e6e6e6', lw=0.7, zorder=0)
    # colourbar for direction
    sm=mpl.cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
    cb=fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.015); cb.set_label('log$_2$FC', fontsize=8); cb.ax.tick_params(labelsize=7)
    # legend for significance
    ax.text(0.995,-0.30,'significance:  *** p<0.001   ** p<0.01   * p<0.05   n.s.   (faded = n.s.)',
            transform=ax.transAxes, ha='right', fontsize=7.5, color='#777')
    plt.tight_layout(); fig.savefig(FIG/fname); plt.show(); print('wrote',FIG/fname)

## Figure 1 — CXCL8 / IL-8

In [ ]:
pub_plot(cx,'CXCL8  (IL-8)',
         'The inflammatory decision point — log$_2$FC across both studies and all butyrate doses',
         'CXCL8_focus_across_conditions.png')

## Figure 2 — HSPA5 (separate; correlated readout)

In [ ]:
hp=gather('HSPA5'); print('HSPA5 contrasts:',len(hp)); print(hp.round(3).to_string())
pub_plot(hp,'HSPA5  (BiP)',
         'The coupled chaperone readout (ρ=0.92 with CXCL8) — shown separately',
         'HSPA5_across_conditions.png', cmap_name='PuOr_r')

## Notes
- A06 contrasts are auto-discovered from disk (cell 0 prints exactly which tables were found
  and used) — no filename guessing, so the ValueError from the `or` idiom is gone too.
- Colour encodes direction (blue→down, red→up for CXCL8); faded bars are non-significant (p≥0.05);
  stars give the p-value tier. Dashed line separates the two studies.
- Fig 2 (HSPA5) is deliberately separate and uses a different palette — it's the correlated
  chaperone, not an IL-8 comparison; the contrast between Fig 1 and Fig 2 is the dissociation.